In [1]:
from parameter.aparameter import AParameter
from strategy.strategy_factory import StrategyFactory
from transformer.transformer import Transformer
from backtester.backtester import Backtester
from processor.server_processor import ServerProcessor
from datetime import datetime, timedelta
from strategy.strategy import Strategy
import pandas as pd
from tqdm import tqdm
from database.adatabase import ADatabase

In [4]:
db = ADatabase("sapling")
market = ADatabase("market")
market.connect()
tickers = market.retrieve("russell1000")["ticker"]
market.disconnect()
names = [x for x in Strategy._member_names_ if "FACTOR_LOADING" not in x]
queries = []
for name in tqdm(names):
    for holding_period in [5]:
        for ascending in [True,False]:
            for positions in [5]:
                for stop_loss in [0.05]:
                    query = {
                        "strategy":name,
                        "holding_period":holding_period,
                        "positions":positions,
                        "stop_loss":stop_loss,
                        "ascending":ascending,
                        "tickers":tickers
                    }
                    queries.append(query)

100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 16/16 [00:00<?, ?it/s]


In [ ]:
analysis = []
db.connect()
db.drop("kpi")
for query in tqdm(queries):
    try:
        parameter = AParameter()
        parameter.build(query)
        start = datetime.now() - timedelta(days=365.25*2)
        end = datetime.now()
        strategy = StrategyFactory.build(parameter)
        simulation = Transformer.local_transform(strategy,start,end)
        trades = Backtester.backtest(strategy,simulation)
        portfolio = Backtester.portfolio(trades)
        recommendations = Backtester.recommendations(trades)
        kpi = Backtester.kpi(trades,portfolio)
        results = ServerProcessor.server_format(strategy,trades,portfolio,recommendations,kpi)
        db.store("kpi",pd.DataFrame([results["kpi"] | query]).drop("tickers",axis=1))
    except Exception as e:
        print(str(e))
        continue
db.disconnect()

 53%|████████████████████████████████████████████████████████████████████████▊                                                                | 17/32 [15:03<13:08, 52.59s/it]